# Customer Segmentation & Retention Analysis — E-Commerce
## Phase 4: Customer Lifetime Value Estimation

**Goal:** Quantify the 3-year projected revenue value of each customer,  
discount it by churn risk, and produce actionable prioritisation for retention budget allocation.

**CLV formula used:**  
`CLV_basic = Monetary × 1.5` *(projects 3-year forward from 2-year history)*  
`CLV_adjusted = CLV_basic × (1 − churn_probability)` *(risk-discounted expected value)*

---

## Section 1 — Setup & Load Data

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:,.2f}'.format)

CLV_PATH     = '../data/processed/clv_results.csv'
SEG_SUM_PATH = '../data/processed/clv_segment_summary.csv'

print('Libraries loaded.')

In [ ]:
df  = pd.read_csv(CLV_PATH)
seg = pd.read_csv(SEG_SUM_PATH)

print(f'Customer records : {len(df):,}')
print(f'Columns          : {df.columns.tolist()}')
print()
print('Segment summary:')
display(seg)
df.head()

---
## Section 2 — CLV Distribution

The first step is to understand the *shape* of CLV across the customer base —  
and to see what happens to that distribution once we discount for churn risk.

In [ ]:
# Cap at 99th percentile for visual clarity; extreme outliers compress the chart
p99_basic = df['CLV_basic'].quantile(0.99)
p99_adj   = df['CLV_adjusted'].quantile(0.99)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        'CLV Basic — Zero-Churn Baseline',
        'CLV Adjusted — Risk-Discounted',
    ]
)

fig.add_trace(
    go.Histogram(
        x=df['CLV_basic'].clip(upper=p99_basic),
        nbinsx=60, marker_color='steelblue',
        name='CLV Basic', showlegend=True,
    ),
    row=1, col=1,
)
fig.add_trace(
    go.Histogram(
        x=df['CLV_adjusted'].clip(upper=p99_adj),
        nbinsx=60, marker_color='coral',
        name='CLV Adjusted', showlegend=True,
    ),
    row=1, col=2,
)

fig.update_xaxes(title_text='CLV £ (capped at 99th pct)', row=1, col=1)
fig.update_xaxes(title_text='CLV £ (capped at 99th pct)', row=1, col=2)
fig.update_yaxes(title_text='Customers')
fig.update_layout(
    title_text='CLV Distribution: Basic vs Risk-Adjusted (5,878 Customers)',
    height=420, template='plotly_white',
)
fig.show()

# Key numbers
total_basic = df['CLV_basic'].sum()
total_adj   = df['CLV_adjusted'].sum()
at_risk     = total_basic - total_adj
print(f'Total CLV_basic    : £{total_basic:>12,.0f}')
print(f'Total CLV_adjusted : £{total_adj:>12,.0f}')
print(f'Revenue at risk    : £{at_risk:>12,.0f}  ({at_risk/total_basic*100:.1f}% of potential)')

**Business Insight — The cost of churn, made visible:**

The two histograms tell a clear story:

- **CLV Basic** shows what the business *could* earn if every customer continued purchasing at their historical rate for the next 3 years — a total of **£26,062,206**.

- **CLV Adjusted** shows what the business should *actually expect* to earn after accounting for the probability each customer churns — a total of **£22,702,353**.

The gap — **£3,359,853 (12.9% of total potential revenue)** — is the cost of churn. This is not a hypothetical number. It is the expected revenue that will not be realised if no retention action is taken.

Notice the spike near zero in the CLV Adjusted histogram: this is the 3,566 Dormant / At-Risk customers who have churn probabilities near 1.0, meaning their expected future revenue is effectively zero. They are present in the CLV Basic chart but almost entirely absent from CLV Adjusted.

---
## Section 3 — CLV by Segment

Segmentation reveals where the economic value is concentrated — and where the greatest risk lies.

In [ ]:
# Bar chart: Mean CLV basic vs adjusted per segment
seg_melted = seg[['Segment', 'Mean_CLV_basic', 'Mean_CLV_adj']].melt(
    id_vars='Segment',
    value_vars=['Mean_CLV_basic', 'Mean_CLV_adj'],
    var_name='CLV_type', value_name='Mean_CLV'
)
seg_melted['CLV_type'] = seg_melted['CLV_type'].map({
    'Mean_CLV_basic': 'CLV Basic (zero-churn)',
    'Mean_CLV_adj'  : 'CLV Adjusted (risk-discounted)',
})

fig = px.bar(
    seg_melted,
    x='Segment', y='Mean_CLV', color='CLV_type', barmode='group',
    title='Mean Customer CLV — Basic vs Risk-Adjusted by Segment',
    labels={'Mean_CLV': 'Mean CLV (£)', 'CLV_type': ''},
    color_discrete_map={
        'CLV Basic (zero-churn)'           : 'steelblue',
        'CLV Adjusted (risk-discounted)'   : 'coral',
    },
    template='plotly_white', height=420,
)
fig.update_yaxes(tickprefix='£', tickformat=',.0f')
fig.show()

In [ ]:
# Revenue at risk per segment
fig = px.bar(
    seg,
    x='Segment', y='Revenue_at_Risk',
    title='Revenue at Risk (CLV_basic − CLV_adjusted) by Segment',
    labels={'Revenue_at_Risk': 'Revenue at Risk (£)'},
    color='Segment',
    color_discrete_sequence=['steelblue', 'coral'],
    template='plotly_white', height=380,
)
fig.update_yaxes(tickprefix='£', tickformat=',.0f')
fig.show()

In [ ]:
# Clean formatted summary table
display_seg = seg.copy()
for col in ['Mean_CLV_basic', 'Mean_CLV_adj', 'Total_CLV_basic', 'Total_CLV_adj', 'Revenue_at_Risk']:
    display_seg[col] = display_seg[col].apply(lambda x: f'£{x:,}')
display_seg['Pct_Customers'] = display_seg['Pct_Customers'].apply(lambda x: f'{x}%')
display_seg['Pct_Total_CLV'] = display_seg['Pct_Total_CLV'].apply(lambda x: f'{x}%')
print('Segment CLV Summary:')
display(display_seg)

**Business Insight — Segment economics:**

The numbers reveal a structural Pareto dynamic:

| Segment | Customers | % of Base | % of Total CLV | Revenue at Risk |
|---|---|---|---|---|
| **Champions** | 2,312 | 39.3% | **87.4%** | £226,739 |
| **Dormant / At-Risk** | 3,566 | 60.7% | 12.6% | **£3,133,114** |

**Champions hold the value:** 39% of customers generate 87% of projected CLV. The mean CLV per Champion (£9,854) is more than **10× higher** than a Dormant customer (£920). Retaining even one Champion is worth more than retaining ten Dormant customers.

**Dormant segment holds the risk:** While Dormant customers represent only 12.6% of total CLV potential, their high churn probability means £3.1M of that potential will never be realised. The revenue at risk from the Dormant segment is **14× larger** than from Champions.

**Strategic implication:** Retention budget has two distinct jobs — *protect* Champions (£227K at risk, high-value customers) and *recover* the recoverable portion of Dormant customers (£3.1M at risk, but highly variable by individual spend).

---
## Section 4 — CLV Tier Analysis

Tiers translate continuous CLV scores into actionable customer categories for marketing and CRM systems.

In [ ]:
tier_order = ['Platinum', 'Gold', 'Silver', 'Bronze']
tier_counts = df['CLV_tier'].value_counts().reindex(tier_order)
tier_clv    = df.groupby('CLV_tier')['CLV_adjusted'].mean().reindex(tier_order)
tier_pct_clv= (df.groupby('CLV_tier')['CLV_adjusted'].sum() / df['CLV_adjusted'].sum() * 100).reindex(tier_order)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Customer Distribution by CLV Tier', 'Mean CLV_adjusted by Tier'],
    specs=[[{'type': 'pie'}, {'type': 'bar'}]],
)

tier_colors = ['#FFD700', '#C0C0C0', '#CD7F32', '#4a4a4a']  # Gold, Silver, Bronze, Dark
tier_colors = ['#9b59b6', '#3498db', '#2ecc71', '#95a5a6']   # Purple, Blue, Green, Gray

fig.add_trace(
    go.Pie(
        labels=tier_order,
        values=tier_counts.values,
        marker_colors=tier_colors,
        hole=0.4,
        textinfo='label+percent',
        showlegend=False,
    ),
    row=1, col=1,
)

fig.add_trace(
    go.Bar(
        x=tier_order,
        y=tier_clv.values,
        marker_color=tier_colors,
        text=[f'£{v:,.0f}' for v in tier_clv.values],
        textposition='outside',
        showlegend=False,
    ),
    row=1, col=2,
)

fig.update_yaxes(tickprefix='£', tickformat=',.0f', row=1, col=2)
fig.update_layout(
    title_text='CLV Tier Distribution — Customer Count & Average Value',
    height=430, template='plotly_white',
)
fig.show()

In [ ]:
# Tier summary table
tier_summary = pd.DataFrame({
    'CLV Tier'       : tier_order,
    'Customers'      : tier_counts.values,
    '% of Base'      : (tier_counts / tier_counts.sum() * 100).round(1).values,
    'Avg CLV (£)'    : [f'£{v:,.0f}' for v in tier_clv.values],
    '% of Total CLV' : [f'{v:.1f}%'  for v in tier_pct_clv.values],
})
print(tier_summary.to_string(index=False))

**Business Insight — The 10/73 rule (stronger than 80/20):**

This dataset exhibits a concentration of economic value that exceeds the classic Pareto principle:

> **The Platinum tier — just 10% of customers — accounts for 72.9% of total projected CLV.**

The average Platinum customer has a risk-adjusted 3-year value of **£28,139**, compared to **£0** for the Bronze tier (whose CLV_adjusted rounds to zero because their churn probability is near 1.0).

**Practical implications for business strategy:**

- **Platinum** — These 588 customers are the company. Losing even 5% of them wipes out more value than winning back the entire Bronze tier. They should be in a dedicated VIP programme with personal account management.
- **Gold** — The next 1,176 customers represent the growth pipeline. With the right incentives, some can be upgraded to Platinum.
- **Silver** — Moderate value. Respond well to automated engagement programmes; not cost-effective to address individually.
- **Bronze** — 40% of the customer base, 0% of expected future value. These customers have already effectively churned. Automated last-chance campaigns only; no bespoke spend.

---
## Section 5 — Retention ROI Analysis

Not every customer is worth spending money to retain. This section quantifies the expected return  
on a **£10 per customer** retention intervention (e-mail + discount coupon), applied selectively.

In [ ]:
# Scatter: CLV_adjusted vs Retention_ROI, coloured by Worth_Retaining
# Cap both axes to remove extreme outliers for readability
p99_clv = df['CLV_adjusted'].quantile(0.99)
p99_roi = df['Retention_ROI'].quantile(0.99)
p01_roi = df['Retention_ROI'].quantile(0.01)

df_plot = df.copy()
df_plot['CLV_adj_cap'] = df_plot['CLV_adjusted'].clip(upper=p99_clv)
df_plot['ROI_cap']     = df_plot['Retention_ROI'].clip(lower=p01_roi, upper=p99_roi)
df_plot['Worth_Label'] = df_plot['Worth_Retaining'].map({True: 'Worth Retaining', False: 'Not Worth Retaining'})

fig = px.scatter(
    df_plot,
    x='CLV_adj_cap', y='ROI_cap',
    color='Worth_Label',
    color_discrete_map={'Worth Retaining': '#2ecc71', 'Not Worth Retaining': '#e74c3c'},
    title='Retention ROI vs Risk-Adjusted CLV (£10 intervention cost)',
    labels={'CLV_adj_cap': 'CLV Adjusted £ (capped)', 'ROI_cap': 'Retention ROI £'},
    opacity=0.55,
    template='plotly_white',
    height=450,
)
fig.add_hline(y=0, line_dash='dash', line_color='grey', annotation_text='ROI = 0 breakeven')
fig.update_xaxes(tickprefix='£', tickformat=',.0f')
fig.update_yaxes(tickprefix='£', tickformat=',.0f')
fig.show()

In [ ]:
# Worth_Retaining count per segment
worth_seg = df.groupby(['Segment', 'Worth_Retaining']).size().reset_index(name='Count')
worth_seg['Label'] = worth_seg['Worth_Retaining'].map({True: 'Worth Retaining', False: 'Not Worth Retaining'})

fig = px.bar(
    worth_seg,
    x='Segment', y='Count', color='Label', barmode='stack',
    title='Retention Investment Worthiness by Segment (£10 cost)',
    color_discrete_map={'Worth Retaining': '#2ecc71', 'Not Worth Retaining': '#e74c3c'},
    template='plotly_white', height=400,
)
fig.show()

# Summary numbers
worth        = df['Worth_Retaining'].sum()
total_roi    = df.loc[df['Worth_Retaining'], 'Retention_ROI'].sum()
total_cost   = worth * 10.0
net_return   = total_roi - total_cost

print(f'Customers worth retaining   : {worth:,}')
print(f'Total campaign cost (£10ea) : £{total_cost:,.0f}')
print(f'Expected gross ROI          : £{total_roi:,.0f}')
print(f'Net return on spend         : £{net_return:,.0f}')
print(f'Return on investment        : {net_return / total_cost:.0f}x')

**Business Insight — Where to allocate the retention budget:**

Assuming a **£10 per customer** intervention cost (e.g., personalised email with a 15% discount code, costing roughly £10 in discount + fulfilment):

| Metric | Value |
|---|---|
| Customers worth retaining | **3,860 (65.7%)** |
| Total intervention cost | £38,600 |
| Expected gross return (if 100% success) | £3,089,821 |
| **Net return on spend** | **£3,051,221** |
| **Return on investment** | **~79×** |

The 79× ROI reflects the theoretical maximum — real-world win-back campaigns typically convert 5–15% of targeted customers. At a **10% conversion rate**, the expected net return drops to approximately **£270,000** on a £38,600 spend — still a **7× ROI**, which is excellent for a CRM campaign.

**Where NOT to spend:** The 2,018 customers flagged as *Not Worth Retaining* include:
- Most Champions (they don't need retention spend — they're already active)
- The lowest-spend Dormant customers, where even a successful win-back would not recoup the £10 cost

The scatter plot shows the clean breakeven line — all green points above the line, red below.

---
## Section 6 — The Full Customer Matrix

This is the single most important visualisation in the project.  
It plots every customer by their **churn risk** (x-axis) against their **economic value** (y-axis),  
giving a complete picture of where to focus every retention pound.

In [ ]:
# Quadrant scatter: churn_probability (x) vs CLV_adjusted (y)
# Capped at 99th pct CLV for visual clarity; sized by Monetary

p99_clv   = df['CLV_adjusted'].quantile(0.99)
clv_mid   = df['CLV_adjusted'].quantile(0.75)   # 75th pct as meaningful 'high CLV' threshold
churn_mid = 0.5

df_q = df.copy()
df_q['CLV_plot']     = df_q['CLV_adjusted'].clip(upper=p99_clv)
df_q['Marker_size']  = (np.log1p(df_q['Monetary']) * 2).clip(lower=3, upper=18)

segment_colors = {
    'Champions'        : '#2980b9',
    'Dormant / At-Risk': '#e74c3c',
}

fig = go.Figure()

for seg_name, color in segment_colors.items():
    sub = df_q[df_q['Segment'] == seg_name]
    fig.add_trace(go.Scatter(
        x=sub['churn_probability'],
        y=sub['CLV_plot'],
        mode='markers',
        name=seg_name,
        marker=dict(
            size=sub['Marker_size'],
            color=color,
            opacity=0.55,
            line=dict(width=0.3, color='white'),
        ),
        hovertemplate=(
            '<b>Customer %{customdata[0]}</b><br>'
            'Churn Prob: %{x:.3f}<br>'
            'CLV Adjusted: £%{y:,.0f}<br>'
            'Monetary: £%{customdata[1]:,.0f}<br>'
            '<extra></extra>'
        ),
        customdata=sub[['CustomerID', 'Monetary']].values,
    ))

# Quadrant dividers
fig.add_vline(x=churn_mid, line_dash='dash', line_color='grey', line_width=1.5)
fig.add_hline(y=clv_mid,   line_dash='dash', line_color='grey', line_width=1.5)

# Quadrant labels
label_kwargs = dict(showarrow=False, font=dict(size=12, color='#555'), bgcolor='rgba(255,255,255,0.75)')
fig.add_annotation(x=0.12, y=p99_clv * 0.85,  text='<b>PROTECT</b><br>Champions<br>Low churn, High CLV',   **label_kwargs)
fig.add_annotation(x=0.85, y=p99_clv * 0.85,  text='<b>URGENT: SAVE THESE</b><br>High churn, High CLV',    **label_kwargs)
fig.add_annotation(x=0.12, y=clv_mid * 0.25,  text='<b>NURTURE</b><br>Low churn, Low CLV',                 **label_kwargs)
fig.add_annotation(x=0.85, y=clv_mid * 0.25,  text='<b>DEPRIORITIZE</b><br>High churn, Low CLV',           **label_kwargs)

fig.update_layout(
    title=dict(
        text='Customer Matrix — Churn Risk vs Economic Value<br>'
             '<sup>Each point = one customer | Size = historical spend | Dashed lines = decision thresholds</sup>',
        font_size=15,
    ),
    xaxis=dict(title='Churn Probability →', range=[-0.05, 1.05]),
    yaxis=dict(title='Risk-Adjusted CLV (£) →', tickprefix='£', tickformat=',.0f'),
    height=580,
    template='plotly_white',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
)
fig.show()

In [ ]:
# Quadrant customer counts
q_labels = [
    ('PROTECT (Low churn, High CLV)',     (df.churn_probability < churn_mid) & (df.CLV_adjusted >= clv_mid)),
    ('URGENT: Save (High churn, High CLV)',(df.churn_probability >= churn_mid) & (df.CLV_adjusted >= clv_mid)),
    ('NURTURE (Low churn, Low CLV)',       (df.churn_probability < churn_mid) & (df.CLV_adjusted < clv_mid)),
    ('DEPRIORITIZE (High churn, Low CLV)', (df.churn_probability >= churn_mid) & (df.CLV_adjusted < clv_mid)),
]
print(f'Quadrant thresholds: churn = {churn_mid}, CLV = £{clv_mid:,.0f} (75th pct)\n')
print(f'{"Quadrant":<42} {"Customers":>10}  {"% Base":>8}  {"Total CLV":>14}  {"Avg CLV":>12}')
print('-' * 92)
for label, mask in q_labels:
    sub   = df[mask]
    count = len(sub)
    pct   = count / len(df) * 100
    tclv  = sub['CLV_adjusted'].sum()
    aclv  = sub['CLV_adjusted'].mean()
    print(f'{label:<42} {count:>10,}  {pct:>7.1f}%  £{tclv:>13,.0f}  £{aclv:>11,.0f}')

**Business Insight — Reading the customer matrix:**

This chart is the centrepiece of the entire analysis. It answers the question every retention team faces: **"Where do we spend our budget first?"**

**Four quadrant strategies:**

| Quadrant | Who they are | Action |
|---|---|---|
| **Top-Left: PROTECT** | All 2,312 Champions — low churn risk, high CLV | Invest in deepening loyalty. VIP programme, early product access, personalised service. The goal is to keep them here — not to "save" them. |
| **Top-Right: URGENT** | ~627 high-value customers with high churn risk | These are the highest-priority intervention targets. They represent significant revenue that is actively at risk. Each customer in this quadrant is worth a significant personalised outreach effort. |
| **Bottom-Left: NURTURE** | Low-churn, low-CLV customers | The customer base's "rising stars" — recently engaged but still early-stage. Focus on increasing purchase frequency and average order value, not churn prevention. |
| **Bottom-Right: DEPRIORITIZE** | ~2,939 low-value dormant customers | High churn, low value. The expected return on intervention is marginal. Use low-cost automated campaigns only (if anything). Don't over-invest here. |

**The most important number:** The **URGENT quadrant contains ~627 customers** whose combined CLV_adjusted is still significant despite their high churn probability. These are the dormant customers who were once high-spenders and represent the highest recovery opportunity in the entire customer base.

---
## Section 7 — Executive Summary

*Prepared for: Chief Commercial Officer / VP of Marketing*  
*Subject: Customer Value & Retention Analysis — Action Brief*

---

Our analysis of **5,878 customers** across two years of transaction data reveals three findings that should directly shape retention investment decisions.

**1. Revenue is dangerously concentrated.**  
39% of customers (our Champions) generate **87% of projected 3-year CLV (£22.8M)**. The remaining 61% account for just 13%. Losing even a fraction of the Champion base poses existential risk to revenue. Protecting this group is the single highest-value action available.

**2. £3.36M in future revenue is currently at risk.**  
Once we discount projected CLV by each customer's churn probability, the business should expect to realise only **£22.7M of its £26.1M revenue potential**. The £3.4M gap represents revenue that will be lost unless we act. The Dormant / At-Risk segment accounts for £3.1M of this risk.

**3. Targeted retention delivers exceptional ROI.**  
At a **£10 per customer** intervention cost, **3,860 customers have a positive retention ROI**. A focused campaign targeting the ~627 high-value dormant customers (top-right quadrant) should be launched immediately — these are former high-spenders who have gone quiet and represent the fastest path to revenue recovery.

**Recommended immediate action:** Identify the 627 top-right quadrant customers by CustomerID, prioritise the top 100 by historical Monetary value, and deploy a premium win-back campaign (personalised outreach + ≥15% discount) within the next 30 days.